In [11]:
#Conform the notebook kernel
import sys

print(sys.executable)
print(sys.version)

/opt/anaconda3/bin/python
3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:11:29) [Clang 20.1.8 ]


In [12]:
import os
from dotenv import load_dotenv

loaded = load_dotenv()
# load_dotenv(override=True) #Helpful for reloading environment locally, but not needed in production

print("Was a .env file found?", loaded)


Was a .env file found? True


In [ ]:
# Check for required environment variables without exposing their values
required_variables = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_CHAT_DEPLOYMENT",
    "AZURE_OPENAI_API_VERSION",
]

for variable in required_variables:
    value = os.getenv(variable)

    if value:
        print(f"{variable}: loaded")
    else:
        print(f"{variable}: missing")

AZURE_OPENAI_API_KEY: loaded
AZURE_OPENAI_ENDPOINT: loaded
AZURE_OPENAI_CHAT_DEPLOYMENT: loaded
AZURE_OPENAI_API_VERSION: loaded


In [ ]:
# Step 1: Define the business problem like defining our domain model in software engineering terms
FEATURES = [
    "Dark Mode",
    "AI-based Smart Search",
    "Offline Support",
]

In [ ]:
# Step 2: Define the three scoring functions for the features
# we want 3 options from each feature
# User Advocate → Do users want it?
# Tech Lead → How feasible is it?
# Business Analyst → Is it valuable to the business?

def user_advocate_tool(feature):
    scores = {
        "Dark Mode": (
            5,
            "Some users want it for comfort"
        ),
        "AI-based Smart Search": (
            9,
            "Highly requested feature"
        ),
        "Offline Support": (
            7,
            "Useful for travel and remote work"
        )
    }

    score, comment = scores.get(feature, (0, "No data"))

    return (
        f"{feature} → User Score: {score}. "
        f"Note: {comment}"
    )


def tech_lead_tool(feature):
    scores = {
        "Dark Mode": (
            8,
            "Easy to implement"
        ),
        "AI-based Smart Search": (
            5,
            "Requires custom model integration"
        ),
        "Offline Support": (
            6,
            "Moderate complexity"
        )
    }

    score, comment = scores.get(feature, (0, "No data"))

    return (
        f"{feature} → Tech Score: {score}. "
        f"Note: {comment}"
    )


def business_analyst_tool(feature):
    scores = {
        "Dark Mode": (
            4,
            "Minimal impact on revenue"
        ),
        "AI-based Smart Search": (
            9,
            "Improves retention and engagement"
        ),
        "Offline Support": (
            6,
            "Moderate business value"
        )
    }

    score, comment = scores.get(feature, (0, "No data"))

    return (
        f"{feature} → Business Score: {score}. "
        f"Note: {comment}"
    )

In [ ]:
# Test the functions
print(user_advocate_tool("Dark Mode"))
print(tech_lead_tool("Dark Mode"))
print(business_analyst_tool("Dark Mode"))

Dark Mode → User Score: 5. Note: Some users want it for comfort
Dark Mode → Tech Score: 8. Note: Easy to implement
Dark Mode → Business Score: 4. Note: Minimal impact on revenue


In [ ]:
## Lets convert those normal Python functions into LangChain tools

# Step 3: Import Tool
from langchain_core.tools import Tool

In [ ]:
# Step 4: Wrap the functions as tools
"""
description

The description tells the LLM:

* what the tool does,
* when it should use it,
* what input it expects.

This matters because the LLM does not inspect and understand the Python implementation directly. It mainly sees the tool’s name, description, and input schema.
"""

tools = [
    Tool(
        name="UserAdvocate",
        func=user_advocate_tool,
        description=(
            "Scores a product feature from the user's perspective, "
            "including desirability, usability, and customer value. "
            "Input must be the exact feature name."
        )
    ),
    Tool(
        name="TechLead",
        func=tech_lead_tool,
        description=(
            "Scores a product feature based on technical feasibility, "
            "implementation effort, and engineering complexity. "
            "Input must be the exact feature name."
        )
    ),
    Tool(
        name="BusinessAnalyst",
        func=business_analyst_tool,
        description=(
            "Scores a product feature based on business value, "
            "revenue impact, retention, and strategic benefit. "
            "Input must be the exact feature name."
        )
    )
]


In [ ]:
# Step 5: Inspect the registered tools

for tool in tools:
    print(f"Name: {tool.name}")
    print(f"Description: {tool.description}")
    print("-" * 60)

Name: UserAdvocate
Description: Scores a product feature from the user's perspective, including desirability, usability, and customer value. Input must be the exact feature name.
------------------------------------------------------------
Name: TechLead
Description: Scores a product feature based on technical feasibility, implementation effort, and engineering complexity. Input must be the exact feature name.
------------------------------------------------------------
Name: BusinessAnalyst
Description: Scores a product feature based on business value, revenue impact, retention, and strategic benefit. Input must be the exact feature name.
------------------------------------------------------------


In [ ]:
# Step 6: Test one wrapped tool
result = tools[0].invoke("Dark Mode")
print(result)

Dark Mode → User Score: 5. Note: Some users want it for comfort


In [23]:
# Step 7: Import AzureChatOpenAI
import os
from langchain_openai import AzureChatOpenAI

In [24]:
# Step 8: Create the GPT-5 Mini client
llm = AzureChatOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

In [25]:
# Step 9: Test the LLM before binding tools
response = llm.invoke(
    "Reply with exactly: Azure OpenAI connection successful"
)

print(response.content)

Azure OpenAI connection successful


In [ ]:
# Step 10: Bind the tools
#        This creates a new model object that knows about the three tools.
llm_with_tools = llm.bind_tools(tools)

In [29]:
# # Step 11: Confirm the model can request a tool

from langchain_core.messages import HumanMessage

test_response = llm_with_tools.invoke(
    [
        HumanMessage(
            content=(
                "Use the UserAdvocate tool to evaluate Dark Mode. "
                "Do not answer from your own knowledge."
            )
        )
    ]
)

print("Content:")
print(test_response.content)

print("\nTool calls:")
print(test_response.tool_calls)

Content:


Tool calls:
[{'name': 'UserAdvocate', 'args': {'__arg1': 'Dark Mode'}, 'id': 'call_h2MKGt83VIskgbjxV3RGnbtv', 'type': 'tool_call'}]


In [31]:
# Step 12: Import ToolMessage
from langchain_core.messages import ToolMessage

In [35]:
# Step 13: Create the run_agent() loop
"""Ask model:
→ model requests tool
→ Python runs tool
→ return result to model
→ model may request another tool
→ repeat
→ final answer
"""
def run_agent(prompt: str):
    messages = [
        HumanMessage(content=prompt) #This stores the conversation history.
    ]

    while True:
        response = llm_with_tools.invoke(messages) #The model reads the full history and decides what to do next.
        messages.append(response)

        if not response.tool_calls: #Check whether it wants a tool
            return response.content #If there are no more tool calls, the model has finished and returned the final recommendation

        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call.get("args", {})

            selected_tool = next( #Find the requested tool
                tool for tool in tools
                if tool.name == tool_name
            )

            if "__arg1" in tool_args:
                tool_output = selected_tool.invoke(
                    tool_args["__arg1"]
                )
            else:
                tool_output = selected_tool.invoke(
                    tool_args
                )

            messages.append(
                ToolMessage( #Return the result to GPT-5 Mini
                    content=str(tool_output),
                    tool_call_id=tool_call["id"] #The tool_call_id connects the tool result to the exact request made by the model.
                )
            )

In [36]:
# Step 14: Test the complete loop with one feature
test_result = run_agent(
    """
    Evaluate Dark Mode using all three available tools.

    Return:
    - User Score
    - Technical Feasibility Score
    - Business Value Score
    - Total Score
    """
)

print(test_result)

- User Score: 5
- Technical Feasibility Score: 8
- Business Value Score: 4
- Total Score: 17


In [37]:
# Final assignment step: evaluate all three features
prompt = """
Evaluate each of the following features one by one:

- Dark Mode
- AI-based Smart Search
- Offline Support

For each feature, use all three available tools to get:

- User Score from UserAdvocate
- Technical Feasibility Score from TechLead
- Business Value Score from BusinessAnalyst

Then:

1. Add the three scores for each feature.
2. Calculate the average score for each feature.
3. Return a prioritized list from highest score to lowest.
4. Briefly explain why each feature received its ranking.
"""

result = run_agent(prompt)

print(result)

Results (scores from tools: UserAdvocate = User Score, TechLead = Tech Feasibility Score, BusinessAnalyst = Business Value Score)

1) Dark Mode
- User Score: 5 (Some users want it for comfort)
- Tech Feasibility Score: 8 (Easy to implement)
- Business Value Score: 4 (Minimal impact on revenue)
- Total: 17
- Average: 5.67

2) AI-based Smart Search
- User Score: 9 (Highly requested feature)
- Tech Feasibility Score: 5 (Requires custom model integration)
- Business Value Score: 9 (Improves retention and engagement)
- Total: 23
- Average: 7.67

3) Offline Support
- User Score: 7 (Useful for travel and remote work)
- Tech Feasibility Score: 6 (Moderate complexity)
- Business Value Score: 6 (Moderate business value)
- Total: 19
- Average: 6.33

Prioritized list (highest average → lowest)
1. AI-based Smart Search — Total 23, Avg 7.67
   - Why: Strong user demand and high business impact (retention/engagement) outweigh the moderate technical complexity. Delivers differentiated value and potent